# Data Analysis Agent Exploration

Learn the workflow one small cell at a time: create data, load it, inspect it, and ask an analytical question.

In [1]:
import random

print("random imported")

random imported


In [2]:
from datetime import date, timedelta

print("date helpers imported")

date helpers imported


In [ ]:
import os

print("os imported")

In [ ]:
from pathlib import Path

print("Path imported")

In [ ]:
from dotenv import load_dotenv

print("dotenv imported")

In [ ]:
# Load the .env file from the agent folder.
agent_folder = Path.cwd().parent
load_dotenv(agent_folder / ".env")

print(f"OPENAI_API_KEY configured: {bool(os.getenv('OPENAI_API_KEY'))}")

In [5]:
import pandas as pd

print("pandas imported")

pandas imported


## 1. Create sample data

In [6]:
def create_sample_data(row_count: int = 20) -> pd.DataFrame:
    """Create a small deterministic sales DataFrame."""

    random.seed(42)
    rows = []
    for _ in range(row_count):
        quantity = random.randint(1, 10)
        unit_price = round(random.uniform(50, 500), 2)
        rows.append({
            "date": date(2024, 1, 1) + timedelta(days=random.randint(0, 30)),
            "product": random.choice(["Laptop", "Phone", "Tablet"]),
            "region": random.choice(["North", "South", "West"]),
            "quantity": quantity,
            "unit_price": unit_price,
            "revenue": round(quantity * unit_price, 2),
        })
    return pd.DataFrame(rows)

In [7]:
dataframe = create_sample_data()
print(dataframe.head())
print(f"Shape: {dataframe.shape}")

         date product region  quantity  unit_price  revenue
0  2024-01-09  Laptop  North         2       61.25   122.50
1  2024-01-22  Tablet   West         3      381.41  1144.23
2  2024-01-02  Laptop  North         2      315.72   631.44
3  2024-01-20  Laptop   West         4      154.70   618.80
4  2024-01-23  Tablet  South         4      372.21  1488.84
Shape: (20, 6)


## 2. Validate and inspect data

In [8]:
def validate_dataframe(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Reject empty or oversized DataFrames."""

    if dataframe.empty:
        raise ValueError("DataFrame is empty.")
    if len(dataframe) > 100_000:
        raise ValueError("DataFrame is too large.")
    return dataframe

In [9]:
validated_data = validate_dataframe(dataframe)
print(validated_data.describe(include="all"))

              date product region   quantity  unit_price      revenue
count           20      20     20  20.000000   20.000000    20.000000
unique          14       3      3        NaN         NaN          NaN
top     2024-01-09  Laptop  North        NaN         NaN          NaN
freq             2       8      8        NaN         NaN          NaN
mean           NaN     NaN    NaN   5.150000  233.921500  1054.110500
std            NaN     NaN    NaN   2.978431  121.328232   519.881915
min            NaN     NaN    NaN   1.000000   61.250000   122.500000
25%            NaN     NaN    NaN   2.750000  145.755000   628.280000
50%            NaN     NaN    NaN   4.000000  192.520000  1184.930000
75%            NaN     NaN    NaN   7.500000  336.135000  1438.537500
max            NaN     NaN    NaN  10.000000  460.910000  1843.640000


## 3. Build the analyzer

This cell requires an OpenAI key and explicitly enables generated Python execution. Use only trusted data.

In [10]:
def build_analyzer(dataframe: pd.DataFrame, allow_dangerous_code: bool = False):
    """Create the pandas agent only after explicit consent."""

    if not allow_dangerous_code:
        raise PermissionError("Explicit dangerous-code consent is required.")
    from langchain_experimental.agents import create_pandas_dataframe_agent
    from langchain_openai import ChatOpenAI
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    return create_pandas_dataframe_agent(
        model,
        dataframe,
        verbose=False,
        allow_dangerous_code=True,
    )

In [11]:
try:
    build_analyzer(dataframe)
except PermissionError as error:
    print(f"Safety check: {error}")

Safety check: Explicit dangerous-code consent is required.


## 4. Ask an analytical question

In [12]:
def ask_data(analyzer, question: str) -> str:
    """Send a natural-language question to the pandas agent."""

    if not question.strip():
        raise ValueError("Question cannot be empty.")
    return str(analyzer.invoke({"input": question.strip()})["output"])

print("Question function is ready.")

Question function is ready.


In [17]:
# Test 1: empty questions should be rejected.
try:
    ask_data(None, "   ")
except ValueError as error:
    print(f"Validation passed: {error}")

Validation passed: Question cannot be empty.


## 5. Run a real analysis question

This optional cell creates the analyzer and calls OpenAI. Run it only with trusted data and an `OPENAI_API_KEY` configured.

In [18]:
import os

In [20]:
# Test 2: run one real question after explicit consent.
if "OPENAI_API_KEY" not in os.environ:
    print("Skipped: OPENAI_API_KEY is not configured.")
else:
    analyzer = build_analyzer(dataframe, allow_dangerous_code=True)
    question = "What is the total revenue by product?"
    print(ask_data(analyzer, question))

Skipped: OPENAI_API_KEY is not configured.
